<a href="https://colab.research.google.com/github/lucyford785/Corpus-Linguistics-Data-Collection/blob/main/Data_Collection_Corpus_Linguistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import BNC files:


In [2]:
from google.colab import files
uploaded_files=files.upload()

Saving A0A.xml to A0A.xml
Saving D94.xml to D94.xml


#some test cells

In [ ]:
# #view file:
# with open('A00.xml') as file:
#   test_file=file.read()

# print(test_file)

<bncDoc xml:id="A00"><teiHeader><fileDesc><titleStmt><title> [ACET factsheets &amp; newsletters]. Sample containing about 6688 words of miscellanea (domain: social science) </title><respStmt><resp> Data capture and transcription </resp><name> Oxford University Press </name> </respStmt></titleStmt><editionStmt><edition>BNC XML Edition, December 2006</edition></editionStmt><extent> 6688 tokens; 6708 w-units; 423 s-units </extent><publicationStmt><distributor>Distributed under licence by Oxford University Computing Services on behalf of the BNC Consortium.</distributor><availability> This material is protected by international copyright laws and may not be copied or redistributed in any way. Consult the BNC Web Site at http://www.natcorp.ox.ac.uk for full licencing and distribution conditions.</availability><idno type="bnc">A00</idno><idno type="old"> AidFct </idno></publicationStmt><sourceDesc><bibl><title> [ACET factsheets &amp; newsletters]. </title> <imprint n="AIDSCA1"><publisher> Ai

(For now) extract text from xml tags:

In [ ]:
text_only=re.sub(r'<.*?>', "", test_file)
#? added in regex to prevent entire text being removed - only matches one instance of text between <>
#closest possible following tag - lazy match

In [ ]:
print(text_only)

 [ACET factsheets &amp; newsletters]. Sample containing about 6688 words of miscellanea (domain: social science)  Data capture and transcription  Oxford University Press  BNC XML Edition, December 2006 6688 tokens; 6708 w-units; 423 s-units Distributed under licence by Oxford University Computing Services on behalf of the BNC Consortium. This material is protected by international copyright laws and may not be copied or redistributed in any way. Consult the BNC Web Site at http://www.natcorp.ox.ac.uk for full licencing and distribution conditions.A00 AidFct  [ACET factsheets &amp; newsletters].   Aids Care Education &amp; Training   London   1991-09   1991-09 W nonAc: medicine Health  Sex Tag usage updated for BNC-XMLLast check for BNC World first releaseRedo tagusage tablesCheck all tagcountsResequenced s-units and added headersAdded date infoUpdated all catrefsManually updated tagcounts, titlestmt, and title in sourcePOS codes revised for BNC-2; header updatedInitial accession to cor

In [ ]:
#temp: view text of list of text files to find a spoken example:

#Extract Metadata:

To do: split metadata into spoken and written modes

In [ ]:
#make into a function - cleans up KWIC code?
# def metadata(uploaded_files):
#   "Extract metadata for each hit/instance of 'help'"

#for both modes:
file_id = []
mode = []
year= []
variety=[]
corpus=[]

#for written only:
genre = []
subgenre = []

#for spoken only:
speaker_id=[]
speaker_gender=[]
speaker_age=[]
soc=[]


for filename, content in uploaded_files.items():
    tree = ET.parse(io.BytesIO(content))
    #Sebastian:
    #turning the content into a file-like object for parsing,
    #works in both local and Colab environments

    id = filename.replace('.xml', '')
    file_id.append(id)

    # Find the creation element and get its date attribute
    creation = tree.find('.//creation')
    # gets the value of creation's date attribute
    yearvalue = creation.get('date')
    if yearvalue == '0000':
        yearvalue = '1990'
    year.append(yearvalue)

    #Lucy: added two extra variables from instructions
    variety.append('BrE')
    corpus.append('BNC')

    #Lucy: these are all common between spoken and written mode
    print("file_id:", file_id)
    print("year:", year)
    print("variety:", variety)
    print("corpus:", corpus)

    # Find the classCode element
    classcode = tree.find('.//classCode')
    # Get the full classCode like "W newsp other: science"
    meta = classcode.text

    #Lucy: from here metadata splits into written and sep
    # meta[0] will contain the first character, W or S
    if meta[0]=="W:":
      mode.append("Written")

      #Genre:
      # Loop to find the wtext or stext tags:
      # Try wtext first
      text_element = tree.find('.//wtext')
      if text_element is None:
      # If no wtext, try stext
       text_element = tree.find('.//stext')
      genrevalue = text_element.get('type')
      genre.append(genrevalue)

      # meta[2:] contains everything after W/S and whitespace, add to genre
      subgenre.append(meta[2:])

      print("genre:", genre)
      print("subgenre:", subgenre)


    elif meta[0] == "S":
      mode.append("Spoken")

      #different for spoken:

      #participant data: similar to date, find person element and get its xml:id attribute
      person=tree.find('.//person')
      speaker_id.append(person.get('xml:id'))
      #these 3 all output a code e.g. 'u' - do we need to decode these?
      speaker_gender.append(person.get('sex'))
      speaker_age.append(person.get('ageGroup'))
      soc.append(person.get('soc'))

      # print("speaker_id:", speaker_id)
      # print("speaker_gender:", speaker_gender)
      # print("speaker_agegroup:", speaker_age)
      # print("social_class:", soc)


    # else:
    #   mode.append("Unknown") - does this ever happen?
    # print("mode:", mode)

file_id: ['A00']
year: ['1991']
variety: ['BrE']
corpus: ['BNC']
mode: []


Extract POS tags for all instances of 'help':

In [ ]:

help_instances = []

for filename, content in uploaded_files.items():
    tree = ET.parse(io.BytesIO(content))

    # Find all words
    words = tree.findall('.//w')  # remember w is the XML BNC's tag for word

    # For each instance, get the actual word form and its pos-tag
    for word in words:
      if word.text and word.text.lower().startswith('help'):
        actual_word = word.text
        pos_tag = word.get('c5')  #use c5 not pos because c5 is more detailed
        help_instances.append((filename, actual_word, pos_tag))

# Print what we found
for file, word, pos in help_instances:
    print(f"File: {filename}, Word form: {word}, POS: {pos}")

File: A00.xml, Word form: Helpline , POS: NN1
File: A00.xml, Word form: help , POS: VVI
File: A00.xml, Word form: help , POS: VVI
File: A00.xml, Word form: help , POS: NN1-VVB
File: A00.xml, Word form: help , POS: VVI
File: A00.xml, Word form: help , POS: VVI
File: A00.xml, Word form: help , POS: VVI
File: A00.xml, Word form: helpful , POS: AJ0
File: A00.xml, Word form: help , POS: NN1
File: A00.xml, Word form: help , POS: NN1-VVB
File: A00.xml, Word form: helping , POS: VVG
File: A00.xml, Word form: Help , POS: NN1-VVB
File: A00.xml, Word form: help , POS: NN1
File: A00.xml, Word form: help , POS: VVB-NN1
File: A00.xml, Word form: help , POS: VVI
File: A00.xml, Word form: help , POS: NN1
File: A00.xml, Word form: help , POS: NN1
File: A00.xml, Word form: help , POS: NN1
File: A00.xml, Word form: help , POS: NN1
File: A00.xml, Word form: Help , POS: NN1-VVB
File: A00.xml, Word form: help , POS: NN1
File: A00.xml, Word form: help , POS: VVB-NN1
File: A00.xml, Word form: Help , POS: NN1-

#Generate KWIC concordances:

In [26]:
import io
import re
import spacy
import xml.etree.ElementTree as ET
import pandas as pd

#for both modes:
file_id = []
mode = []
year= []
variety=[]
corpus=[]

#for written only:
genre = []
subgenre = []

#for spoken only:
speaker_id=[]
speaker_gender=[]
speaker_age=[]
soc=[]

# this list will store our KWIC concordance line
# remember: 240 characters before, 480 characters after "help" are required
help_KWIC = []
# this list will store the form of the hit, "help", "helped", "helping" etc.
help_form = []
# this list will store the POS of each hit, noun, verb, adjective, etc.
help_pos = []

hits = []
hit = 1 #this is an index

# We will keep a "buffer" for the context left of "help"
# We go through the xml structure, and record every single word
# but shorten the left context / keep the length of the left context buffer the same

max_buffer = 30

for file, content in uploaded_files.items():
    tree = ET.parse(io.BytesIO(content))

    #Extract metadata (for each file):
    id = filename.replace('.xml', '')
    file_id.append(id)

    # Find the creation element and get its date attribute
    creation = tree.find('.//creation')
    # gets the value of creation's date attribute
    yearvalue = creation.get('date')
    if yearvalue == '0000':
        yearvalue = '1990'
    year.append(yearvalue)

    #Lucy: added two extra variables from instructions
    variety.append('BrE')
    corpus.append('BNC')

    # Find the classCode element
    classcode = tree.find('.//classCode')
    # Get the full classCode like "W newsp other: science"
    meta = classcode.text

    #Lucy: from here metadata splits into written and sep
    # meta[0] will contain the first character, W or S
    if meta[0]=="W:":
      mode.append("Written")

      #Genre:
      # Loop to find the wtext or stext tags:
      # Try wtext first
      text_element = tree.find('.//wtext')
      if text_element is None:
      # If no wtext, try stext
       text_element = tree.find('.//stext')
      genrevalue = text_element.get('type')
      genre.append(genrevalue)

      # meta[2:] contains everything after W/S and whitespace, add to genre
      subgenre.append(meta[2:])

      #set all spoken variables to "NA"
      speaker_id.append("NA")
      speaker_gender.append("NA")
      speaker_age.append("NA")
      soc.append("NA")

    elif meta[0] == "S":
      mode.append("Spoken")

      #participant data: similar to date, find person element and get its xml:id attribute
      person=tree.find('.//person')
      speaker_id.append(person.get('xml:id'))
      #these 3 all output a code e.g. 'u' - do we need to decode these?
      speaker_gender.append(person.get('sex'))
      speaker_age.append(person.get('ageGroup'))
      soc.append(person.get('soc'))

      #set all written variables to "NA"
      genre.append("NA")
      subgenre.append("NA")

    #Generate KWIC Concordances:

    # the left_context will keep track of the 10 previous words - Lucy: previous 30 words as per instructions?
    left_context = []

    # The following parameters are for collecting context to the right of "help"
    collecting_right = False   # unless we find "help", we don't collect right context by default
    right_context = []
    current_right_context_words = 0

    # iterate through every word of the tree
    for elem in tree.iter():

      # If we're collecting right context (after "help")
      if collecting_right:
          if elem.tag in ['w', 'c']:
              right_context.append(elem.text)
              if elem.tag == 'w':  # only count words, not punctuation
                  current_right_context_words += 1
                  if current_right_context_words >= 60:  # stop after ~20 words - 60 words after help?
                      collecting_right = False  # stop collecting
                      right_KWIC = ' '.join(filter(None, right_context)) # changed to filter(None, ...) to avoid None if some elem.text is None
                      # Create KWIC line with left, help word, and right context
                      kwic_line = f"{left_KWIC} @{help_word}@ {right_KWIC}"
                      help_KWIC.append(kwic_line)


      # If the element is a word or punctuation, add it to the left context buffer
      #相当于if elem.text is not None:
      if elem.tag in ['w', 'c'] and elem.text and not elem.text.lower().startswith('help'):
        left_context.append(elem.text)
        # Keep buffer at max size
        if len(left_context) > max_buffer:
            left_context.pop(0)

      # Check if the current elem is a hit of the word "help"
      if elem.tag == 'w' and elem.text and elem.text.lower().startswith('help'):
        # This is an instance of "help"!

        # 新增：如果上一个help还没收集完右侧context，先保存
        if collecting_right:
           right_KWIC = ' '.join(filter(None, right_context))
           help_KWIC.append(f"{left_KWIC} @{help_word}@ {right_KWIC}")
        # 新增


        left_KWIC = ' '.join(filter(None, left_context)) # changed to filter(None, ...) to avoid None if some elem.text is None

        # Get the text of this instance of "help", store it in help_form
        help_word = elem.text
        help_form.append(help_word)
         # get the pos tag (CLAWS-5 c5, not POS)
        help_c5 = elem.get('c5')
        help_pos.append(help_c5)

        # Add the current hit to the hits list and increment it
        hits.append(str(hit))
        hit += 1

        # we have found - so start collecting the right context
        collecting_right = True #这里会将整个loop带回上面的if collect right
        right_context = []  # Reset right context
        current_right_context_words = 0  # Reset word counter

    if collecting_right:
        right_KWIC = ' '.join(filter(None, right_context))
        help_KWIC.append(f"{left_KWIC} @{help_word}@ {right_KWIC}")
        collecting_right = False

# output = "Hit\tKWIC\tHelpForm\tPOS\tFileID\tYear\tVariety\tMode\tCorpus\tSpeakerID\tSpeakerAge\tSpeakerGender\tSocialClass\tGenre\tSubgenre\n"
# for h, kwic, f, pos, id, date, var, mode, corp, sp_id, sp_age, sp_gender, soc, gen, subgen in zip(hits, help_KWIC, help_form, help_pos, file_id, year, variety, mode, corpus, speaker_id, speaker_age, speaker_gender, soc, genre, subgenre):
#   output += f'{h}\t{kwic}\t{f}\t{pos}\t{id}\t{date}\t{var}\t{mode}\t{corp}\t{sp_id}\t{sp_age}\t{sp_gender}\t{soc}\t{genre}\t{subgenre}\n'
# print(output)


# # Save

# with open("BNC_partial_results.txt", "w") as f:
#     f.write(output)

#Split into written and spoken data:
#FIX: hit counter will no longer be consecutive, so separate hits into written and spoken

#index across examples in lists:
for h, kwic, f, pos, id, date, var, mode, corp, sp_id, sp_age, sp_gender, soc, gen, subgen in zip(hits, help_KWIC, help_form, help_pos, file_id, year, variety, mode, corpus, speaker_id, speaker_age, speaker_gender, soc, genre, subgenre):
  if mode=="Written":

    wr_df = pd.DataFrame({
        'Hit': hits,
        'KWIC': help_KWIC,
        'Form': help_form,
        'POS': help_pos,
        'File': file_id,
        'Year': year,
        'Variety': variety,
        'Genre': genre,
        'Subgenre': subgenre,
        'Mode': mode,
        'Corpus': corpus
    })

  elif mode=="Spoken":

     sp_df = pd.DataFrame({
        'Hit': hits,
        'KWIC': help_KWIC,
        'Form': help_form,
        'POS': help_pos,
        'File': file_id,
        'Year': year,
        'Variety': variety,
        'SpeakerID': speaker_id,
        'SpeakerAge': speaker_age,
        'SpeakerGender': speaker_gender,
        'SocialClass': soc,
        'Mode': mode,
        'Corpus': corpus
    })


# wr_df.to_excel('BNC_Written_help_results.xlsx', index=False)
# sp_df.to_excel('BNC_Spoken_help_results.xlsx', index=False)
# #print(f"已保存 {len(df)} 条结果到 BNC_help_results.xlsx")

ValueError: All arrays must be of the same length

#Alternative KWIC concordances using regex?

In [14]:
#Initialising nlp pipeline
import re
import nltk
from nltk.tokenize import word_tokenize
import spacy
nltk.download('punkt')
nltk.download('punkt_tab')
!python -m spacy download en_core_web_lg
nlp = spacy.load('en_core_web_lg')
nlp.max_length = 2000000

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [19]:
hit_counter = 1
corpus = "BNC"
results = "Hit\tDepVar\tHelpPOS\tVoice\tHorrorAequi\tPolarity\tVerbLemma\tObjPresent\tObjectLength\tIntervWords\tGenre\tVariety\tKWIC\tCorpus\n"

for filename in list(uploaded_files):#.keys())[:1]:
  with open (filename, 'r', encoding = 'utf8') as textfile:
    print("Now processing", filename)
    text_content = textfile.read()
    genre = filename.split('-')[0]
    genre = genre.replace('.txt', '')

    text_content = text_content.replace ('\n', ' ')
    text_content = text_content.replace('^', ' ')
    text_content = text_content.replace("*+", "£")
    pattern = r'\*<\*\d+'
    text_content = re.sub(pattern, "", text_content)
    pattern = r'\*\d+'
    text_content = re.sub(pattern, "", text_content)
    text_content_tokens = word_tokenize(text_content)
    text_content = " ".join(text_content_tokens)

    print("Tagging")
    tagged = nlp(text_content)

    string = ""
    for sentence in tagged.sents:
      for token in sentence:
        if not token.is_space:
          string += f'{token.tag}_{token.text}_{token.lemma}_'
    tagged_words = string.split()

    print("C: Finding all instances of 'help' and labelling")
    help_pattern = r'.*_help\w*\b'
    help_positions = []
    for i, word in enumerate(tagged_words):
        if re.search(help_pattern, word, re.IGNORECASE):
            help_positions.append(i)
    help_count = len(help_positions)
    for pos in help_positions:
        tagged_help_word = tagged_words[pos]
        help_parts = tagged_help_word.split('_')
        help_tag = help_parts[0]
        help_form = help_parts[1]
        left_start = max(0, pos - 240)
        left_tokens = tagged_words[left_start:pos]
        right_end = min(len(tagged_words), pos + 480)
        right_tokens = tagged_words[pos+1:right_end]
        left = ' '.join(left_tokens)
        right = ' '.join(right_tokens)
        left_clean = ' '.join([token.split('_')[1] if len(token.split('_')) > 1 else token for token in left_tokens])
        right_clean = ' '.join([token.split('_')[1] if len(token.split('_')) > 1 else token for token in right_tokens])
        kwic_line = f"{left_clean} @{help_form}@ {right_clean}"

        n = 15
        found_to = False
        found_bare = False
        found_ing = False
        found_in = False
        found_verb = False
        dep_var = "NA"
        verb_after_help = "NA"
        intervening_words = "NA"
        found_object = False
        object_presence = "NA"
        object_length = 0
        # Initialize voice, preceding_to, and polarity for each instance of 'help'
        voice = "NA"
        preceding_to = "NA"
        polarity = "NA"

        for i in range(1, n+1):
            if pos + i >= len(tagged_words):
                break
            next_tagged_word = tagged_words[pos + i]
            parts = next_tagged_word.split('_')
            next_tag = parts[0]
            next_word = parts[1]
            next_lemma = parts[2]
            if not help_tag.startswith('VB'):
                break
            if next_tag in ['.', ',', '``', 'HYPH', ':', ")"]:
                break
            if next_tag in ["VBP", "VBD", "VBZ"]:
                break
            if next_tag.startswith('MD') or (next_tag == 'IN' and next_word == 'that') or next_tag.startswith('WP'):
                break
            if next_word.lower() in ['and', 'or', 'are', 'if']:
                break
            if not found_verb:
                if (next_tag in ['NN', 'NP', 'NNS', 'NNP', 'NNPS', 'DT', 'CD', 'WP', 'PRP', 'PRP$'] or
                    next_word.lower() in ['someone', 'anyone', 'who', 'myself', 'this', 'that']):
                    found_object = True
            if found_object:
                if not (next_word.lower() in ['to'] or next_tag in ['RB', 'TO', 'VB']):
                    object_length += 1
            if not found_bare and not found_in and not found_ing and (next_tag == 'TO' or next_word.lower() == 'to'):
                for j in range(1, 3):
                    if pos + i + j < len(tagged_words):
                        potential_verb = tagged_words[pos + i + j]
                        verb_parts = potential_verb.split('_')
                        verb_tag = verb_parts[0]
                        verb_word = verb_parts[1]
                        verb_lemma = verb_parts[2]
                        if verb_tag in ['VB', 'HV', 'BE']:
                            found_to = True
                            verb_after_help = verb_lemma
                            found_verb = True
                            intervening_words = i + j - 2
                            break
            elif not found_to and not found_ing and not found_bare and next_word.lower() == 'in':
                for j in range(1, 3):
                    if pos + i + j < len(tagged_words):
                        potential_ing = tagged_words[pos + i + j]
                        ing_parts = potential_ing.split('_')
                        ing_tag = ing_parts[0]
                        ing_word = ing_parts[1]
                        ing_lemma = ing_parts[2]
                        if (ing_tag in ['VBG', 'HVG', 'BEG'] or ing_word.endswith('ing')):
                            found_in = True
                            verb_after_help = ing_lemma
                            found_verb = True
                            break
            elif not found_to and not found_in and not found_bare and next_tag in ['VBG', 'HVG', 'BEG'] and next_word not in ["according"]:
                found_ing = True
                verb_after_help = next_lemma
                found_verb = True
                intervening_words = i - 1
                break
            elif not found_to and not found_in and not found_ing and next_tag in ['VB', 'HV', 'BE']:
                found_bare = True
                verb_after_help = next_lemma
                found_verb = True
                intervening_words = i - 1
                break
            if found_verb:
                break
        if found_to:
            dep_var = "TO"
        elif found_bare:
            dep_var = "BARE"
        elif found_in:
            dep_var = "INING"
        elif found_ing:
            dep_var = "ING"
        if found_verb:
            if found_object:
                object_presence = "Yes"
            if not found_object:
                object_presence = "No"
                object_length = "NA"
        else:
            object_presence = "NA"
            object_length = "NA"
        if help_tag.startswith('VB'):
            help_pos = help_tag
        elif help_tag.startswith('NN'):
            help_pos = "NOUN"
        elif help_tag.startswith('JJ'):
            help_pos = "ADJ"
        elif help_tag.startswith('RB'):
            help_pos = "ADV"
        else:
            help_pos = "OTHER"

        ################
        #Polarity, voice, and horror aequi extraction here variables to left
        ################

        #Lucy: assign default values to voice, preceding_to, and polarity:

        voice="active"
        preceding_to="NOtoBefore"
        polarity="positive"

        for i in range(1, 60):
          prev_pos = pos - i
          if prev_pos >= 0:
            prev_word = tagged_words[prev_pos]
            prev_parts = prev_word.split('_')
            prev_tag = prev_parts[0]
            prev_text = prev_parts[1].lower()

            #Checking for non-finite complement
            if not help_tag.startswith('VB'):
                        voice = "NA"
                        preceding_to = "NA"
                        polarity = "NA"
                        break

            #Skipping certain punctuation
            if prev_tag in ['.', ',', '``', 'HYPH', ':', ")"]:
                        voice = "NA"
                        preceding_to = "NA"
                        polarity = "NA"
                        break

            if prev_tag in ['BEZ', 'BED', 'BEN', 'BER', 'BE', 'BEM']:
                        # If help is past participle, it's passive
                        if help_tag == 'VBN':
                            voice = "Passive"
                            break
                    # (ii) Check if word before "help" is to
                        if prev_tag == 'TO' or prev_text == 'to':
                        # Found "to" before help
                          preceding_to = "YEStoBefore"
                          break  # We only need to find one "to"
                    # (iii) Check for polarity (can you see a negative word before "help")
                        if prev_text.lower() in ['not', "n't", 'nor', 'no', 'barely', 'hardly', 'scarcely', 'nobody', 'nothing', 'nowhere']:
                        # Found negation before "help"
                          polarity = "NEG"
                          break

        results += f"{hit_counter}\t{dep_var}\t{help_pos}\t{voice}\t{preceding_to}\t{polarity}\t{verb_after_help}\t{object_presence}\t{object_length}\t{intervening_words}\t{genre}\t{kwic_line}\t{corpus}\n"
        hit_counter += 1

print(results)

Now processing A0A.xml
Tagging
C: Finding all instances of 'help' and labelling
Now processing D94.xml
Tagging
C: Finding all instances of 'help' and labelling
Hit	DepVar	HelpPOS	Voice	HorrorAequi	Polarity	VerbLemma	ObjPresent	ObjectLength	IntervWords	Genre	Variety	KWIC	Corpus
1	NA	OTHER	active	NOtoBefore	positive	NA	NA	NA	NA	A0A.xml	 @<@ 	BNC

